# v3.0.0 PD Baseline Comparisons

Baseline experiments on Bridge2AI v3.0.0 PD selected tasks using **identical 5-fold CV splits**
as the primary fine-tuned AST experiment (`v3_PD_AST_Spectrograms.ipynb`).

**Baselines implemented:**
1. **Frozen AST (Linear Probing)** — freeze backbone, train classification head only
2. **Logistic Regression on spectrogram summary statistics** — no deep learning
3. **ResNet18 (ImageNet pretrained, 1-channel)** — alternative deep architecture
4. **SpecAugment ablation** — fine-tuned AST without time/frequency masking

All baselines use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` at participant level.

In [ ]:
#0 - Environment setup & reproducibility
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"

import torch, random, numpy as np
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import copy, time, json

import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import zoom
from scipy.stats import skew, kurtosis
from scipy import stats as sp_stats
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve, confusion_matrix, accuracy_score)
from tqdm import tqdm

ROOT = Path('/data0/b2ai-voice/3.0.0')
SPEC = ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
PD_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Spec exists: {SPEC.exists()}')
print(f'PD phen exists: {PD_PHEN.exists()}')
print(f'Ctrl phen exists: {CTRL_PHEN.exists()}')

In [ ]:
#2 - Load spectrogram data
pf = pq.ParquetFile(SPEC)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
spec = pd.concat(parts, ignore_index=True)

# Zero-pad participant IDs to 6 digits
spec['participant_id'] = spec['participant_id'].astype(str).str.zfill(6)

print(f'Total recordings: {len(spec)}')
print(f'Unique participants: {spec["participant_id"].nunique()}')
print(f'Unique tasks: {spec["task_name"].nunique()}')

In [ ]:
#3 - Build PD labels from hierarchical phenotype
pd_df = pd.read_csv(PD_PHEN, sep='\t')
ctrl_df = pd.read_csv(CTRL_PHEN, sep='\t')

pd_ids = set(pd_df['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_df['participant_id'].astype(str).str.zfill(6))

# Remove any PD-Control overlap
overlap = pd_ids & ctrl_ids
ctrl_ids_clean = ctrl_ids - overlap
print(f'PD participants: {len(pd_ids)}')
print(f'Control participants (clean): {len(ctrl_ids_clean)}')
print(f'Overlap removed: {len(overlap)}')

# Assign labels
spec['label'] = np.nan
spec.loc[spec['participant_id'].isin(pd_ids), 'label'] = 1
spec.loc[spec['participant_id'].isin(ctrl_ids_clean), 'label'] = 0

# Keep only labeled recordings
data = spec.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
print(f'\nLabeled recordings: {len(data)}')
print(f'PD recordings: {(data["label"]==1).sum()}')
print(f'Control recordings: {(data["label"]==0).sum()}')
print(f'Unique participants: {data["participant_id"].nunique()}')

In [ ]:
#4 - Task selection: same 8 selected tasks as primary experiment
SELECTED_TASKS = [
    'prolonged-vowel',
    'glides-high-to-low',
    'glides-low-to-high',
    'diadochokinesis-pataka',
    'rainbow-passage',
    'picture-description',
    'story-recall',
    'maximum-phonation-time-1',
]

MIN_TIME_FRAMES = 100

data_selected = data[
    (data['task_name'].isin(SELECTED_TASKS)) &
    (data['n_frames'] >= MIN_TIME_FRAMES)
].copy()

print(f'Selected task recordings: {len(data_selected)}')
print(f'PD: {(data_selected["label"]==1).sum()} recordings from '
      f'{data_selected[data_selected["label"]==1]["participant_id"].nunique()} participants')
print(f'Ctrl: {(data_selected["label"]==0).sum()} recordings from '
      f'{data_selected[data_selected["label"]==0]["participant_id"].nunique()} participants')
print(f'Total participants: {data_selected["participant_id"].nunique()}')
print(f'\nPer-task counts:')
print(data_selected.groupby('task_name')['label'].value_counts().unstack(fill_value=0))

In [ ]:
#5 - Process spectrograms
TARGET_SEQ_LEN = 1024

def process_spectrogram(spec_raw, target_len=1024):
    """Process raw spectrogram with reflect padding / center crop."""
    spec = np.stack(spec_raw).astype(np.float32)
    n_mels, time_len = spec.shape
    if time_len < target_len:
        pad_width = target_len - time_len
        spec = np.pad(spec, ((0, 0), (0, pad_width)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

X_list = []
for idx, row in tqdm(data_selected.iterrows(), total=len(data_selected), desc='Processing'):
    processed = process_spectrogram(row['mel_spectrogram'], TARGET_SEQ_LEN)
    X_list.append(processed)

X_raw = np.stack(X_list)
y_raw = data_selected['label'].values
participants_raw = data_selected['participant_id'].values
tasks_raw = data_selected['task_name'].values

print(f'Processed shape: {X_raw.shape}')  # (N, 60, 1024)
print(f'Value range: [{X_raw.min():.2f}, {X_raw.max():.2f}]')

In [ ]:
#6 - Participant-level setup for CV (shared across all baselines)
unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])

print(f'Total participants: {len(unique_participants)}')
print(f'PD: {int(participant_labels.sum())} ({participant_labels.mean():.1%})')
print(f'Control: {int((participant_labels == 0).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Pre-compute fold splits (identical for ALL baselines)
FOLD_SPLITS = list(skf.split(unique_participants, participant_labels))
print(f'\nFold splits precomputed: {len(FOLD_SPLITS)} folds')
for i, (tr, va) in enumerate(FOLD_SPLITS):
    tr_labels = participant_labels[tr]
    va_labels = participant_labels[va]
    print(f'  Fold {i+1}: Train {len(tr)} (PD={tr_labels.sum()}) | Val {len(va)} (PD={va_labels.sum()})')

In [ ]:
#7 - Shared model components and helpers

def resize_spectrogram(spec, target_mel=128, target_time=1024):
    """Resize spectrogram to target dimensions (60 -> 128 mel bins)."""
    current_mel, current_time = spec.shape
    mel_ratio = target_mel / current_mel
    time_ratio = target_time / current_time
    resized = zoom(spec, (mel_ratio, time_ratio), order=1)
    return resized.astype(np.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class SpectrogramDataset(torch.utils.data.Dataset):
    """Dataset for spectrogram-based models with optional SpecAugment."""
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            # Time masking
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            # Frequency masking
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}


def evaluate_fold(model, loader, device):
    """Evaluate model and aggregate recording-level predictions to participant level."""
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs, part_labels = [], []
    for p in unique_parts:
        mask = all_parts == p
        part_probs.append(all_probs[mask].mean())
        part_labels.append(all_labels[mask][0])
    return np.array(part_probs), np.array(part_labels), unique_parts


def compute_fold_metrics(part_probs, part_labels):
    """Compute AUC, F1, recall, precision at optimal Youden threshold."""
    if len(np.unique(part_labels)) < 2:
        return {'auc': 0.5, 'f1': 0.0, 'recall': 0.0, 'precision': 0.0, 'threshold': 0.5}
    auc = roc_auc_score(part_labels, part_probs)
    fpr, tpr, thresholds = roc_curve(part_labels, part_probs)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    preds = (part_probs >= opt_thresh).astype(int)
    return {
        'auc': float(auc),
        'f1': float(f1_score(part_labels, preds, zero_division=0)),
        'recall': float(recall_score(part_labels, preds, zero_division=0)),
        'precision': float(precision_score(part_labels, preds, zero_division=0)),
        'threshold': float(opt_thresh),
    }


def print_cv_summary(name, fold_results, oof_labels, oof_probs):
    """Print CV summary with per-fold results, mean +/- SD, 95% CI, and OOF metrics."""
    print(f'\n{"="*70}')
    print(f'{name} - Per-fold results')
    print(f'{"="*70}')
    for r in fold_results:
        print(f'  Fold {r["fold"]}: AUC={r["auc"]:.4f}  F1={r["f1"]:.4f}  '
              f'Rec={r["recall"]:.4f}  Prec={r["precision"]:.4f}')

    n = len(fold_results)
    t_crit = sp_stats.t.ppf(0.975, df=n - 1)
    print(f'\nMean +/- SD [95% CI]:')
    for metric in ['auc', 'f1', 'recall', 'precision']:
        vals = [r[metric] for r in fold_results]
        m = np.mean(vals)
        sd = np.std(vals, ddof=1)
        se = sd / np.sqrt(n)
        ci_lo = m - t_crit * se
        ci_hi = m + t_crit * se
        print(f'  {metric.upper()}: {m:.4f} ({sd:.4f}) [{ci_lo:.4f}, {ci_hi:.4f}]')

    # OOF metrics
    oof_auc = roc_auc_score(oof_labels, oof_probs)
    fpr, tpr, thresholds = roc_curve(oof_labels, oof_probs)
    opt_idx = np.argmax(tpr - fpr)
    oof_thresh = thresholds[opt_idx]
    oof_preds = (oof_probs >= oof_thresh).astype(int)

    print(f'\nOOF AUC:       {oof_auc:.4f}')
    print(f'OOF F1:        {f1_score(oof_labels, oof_preds, zero_division=0):.4f} '
          f'(threshold={oof_thresh:.3f})')
    print(f'OOF Recall:    {recall_score(oof_labels, oof_preds, zero_division=0):.4f}')
    print(f'OOF Precision: {precision_score(oof_labels, oof_preds, zero_division=0):.4f}')
    return oof_auc


print('Shared helpers defined.')

---
## Baseline 1: Frozen AST (Linear Probing)

Same `ASTClassifier` architecture but with `freeze_base=True`. Only the classification head
(LayerNorm + Linear(768,256) + GELU + Dropout + Linear(256,2)) is trained.
All backbone parameters are frozen.

In [ ]:
#8 - AST model definition (used by Frozen AST and SpecAugment ablation)
from transformers import ASTModel, ASTConfig

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(hidden_size=768, num_hidden_layers=12,
                             num_attention_heads=12, intermediate_size=3072,
                             max_length=1024, num_mel_bins=128)
            self.ast = ASTModel(config)
            hidden_size = 768
        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, 128, 1024) -> (B, 1024, 128)
        outputs = self.ast(input_values=x)
        pooled = outputs.pooler_output
        return self.classifier(pooled)

print('ASTClassifier defined.')

In [ ]:
#9 - Frozen AST: 5-fold CV
frozen_fold_results = []
frozen_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
frozen_oof_labels = participant_labels.astype(np.int64).copy()

total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLD_SPLITS):
    print(f'\n{"="*60}')
    print(f'Fold {fold+1}/{N_FOLDS} (FROZEN AST — Linear Probe)')
    print(f'{"="*60}')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]
    y_train_fold = y_raw[train_mask]
    parts_train_fold = participants_raw[train_mask]

    X_val_fold = X_raw[val_mask]
    y_val_fold = y_raw[val_mask]
    parts_val_fold = participants_raw[val_mask]

    print(f'Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants')
    print(f'Val:   {len(X_val_fold)} recordings from {len(val_parts_fold)} participants')

    # Resize 60 -> 128 mel bins
    X_train_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
    X_val_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

    # Fold-specific z-score normalization
    fold_mean = X_train_ast.mean()
    fold_std = X_train_ast.std()
    X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
    X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)

    # Datasets (augment=True for train, matching primary experiment)
    train_ds = SpectrogramDataset(X_train_ast, y_train_fold, parts_train_fold, augment=True)
    val_ds = SpectrogramDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)

    # Balanced sampler
    class_counts = np.bincount(y_train_fold)
    weights = 1.0 / class_counts
    sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler,
                                               num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False,
                                             num_workers=4, pin_memory=True)

    # Fresh FROZEN model
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=True).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)')

    # Only optimize head params with higher LR (5e-4)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=5e-4, weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights
    cc = np.bincount(y_train_fold)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=class_weights_fold, gamma=2.0)
    print(f'Class weights: {cw}')

    best_score = 0
    best_state = None
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(30):
        model.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels_batch = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        metrics = compute_fold_metrics(part_probs, part_labels_fold)
        score = 0.4 * metrics['auc'] + 0.6 * metrics['f1']

        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '<-- best'
        else:
            patience_counter += 1
            marker = ''

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {metrics["auc"]:.3f} | '
              f'F1 {metrics["f1"]:.3f} {marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and get final predictions
    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        frozen_oof_probs[idx_oof] = part_probs[i]

    # Fold metrics
    fold_metrics = compute_fold_metrics(part_probs, part_labels_fold)
    fold_metrics['fold'] = fold + 1
    frozen_fold_results.append(fold_metrics)

    print(f'Fold {fold+1}: AUC={fold_metrics["auc"]:.4f}, F1={fold_metrics["f1"]:.4f}')

    del model, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

frozen_total_time = time.time() - total_start
print(f'\nTotal Frozen AST CV time: {frozen_total_time/60:.1f} minutes')

# Summary
print_cv_summary('FROZEN AST (LINEAR PROBE)', frozen_fold_results,
                 frozen_oof_labels, frozen_oof_probs)

# Save results
np.savez(str(RESULTS_DIR / 'frozen_ast_v3_cv_results.npz'),
    oof_probs=frozen_oof_probs,
    oof_labels=frozen_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in frozen_fold_results]),
    fold_f1s=np.array([r['f1'] for r in frozen_fold_results]),
    fold_recalls=np.array([r['recall'] for r in frozen_fold_results]),
    fold_precisions=np.array([r['precision'] for r in frozen_fold_results]),
    fold_thresholds=np.array([r['threshold'] for r in frozen_fold_results]),
)
print(f'Saved: {RESULTS_DIR}/frozen_ast_v3_cv_results.npz')

---
## Baseline 2: Logistic Regression on Spectrogram Summary Statistics

For each spectrogram (128x1024 after resize), compute per-frequency-bin statistics
(mean, std, skewness, kurtosis: 128 * 4 = 512 features) plus 5 global features
(overall mean, std, skewness, kurtosis, energy). Total: 517 features.

Recording-level predictions are aggregated to participant level by averaging.

In [ ]:
#10 - Feature extraction from resized spectrograms

def extract_features(spectrogram):
    """Extract summary statistics from a 128x1024 spectrogram.
    
    Returns 517 features:
      - 128 per-bin means
      - 128 per-bin stds
      - 128 per-bin skewness
      - 128 per-bin kurtosis
      - 5 global: mean, std, skewness, kurtosis, energy
    """
    per_bin_mean = spectrogram.mean(axis=1)       # (128,)
    per_bin_std = spectrogram.std(axis=1)          # (128,)
    per_bin_skew = skew(spectrogram, axis=1)       # (128,)
    per_bin_kurt = kurtosis(spectrogram, axis=1)   # (128,)

    global_feats = np.array([
        spectrogram.mean(),
        spectrogram.std(),
        skew(spectrogram.ravel()),
        kurtosis(spectrogram.ravel()),
        (spectrogram ** 2).sum(),  # energy
    ])

    feats = np.concatenate([per_bin_mean, per_bin_std, per_bin_skew, per_bin_kurt, global_feats])
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

# Resize all spectrograms to 128x1024 first, then extract features
print('Resizing spectrograms for feature extraction...')
X_resized = np.stack([resize_spectrogram(x) for x in tqdm(X_raw, desc='Resize', leave=False)])
print(f'Resized shape: {X_resized.shape}')

print('Extracting features...')
features = np.array([extract_features(x) for x in tqdm(X_resized, desc='Extract', leave=False)])
print(f'Feature matrix shape: {features.shape}')
print(f'Features per recording: {features.shape[1]} '
      f'(128 bins x 4 stats = {128*4} + 5 global = {128*4+5})')
print(f'NaN count: {np.isnan(features).sum()}, Inf count: {np.isinf(features).sum()}')

In [ ]:
#11 - Logistic Regression: 5-fold CV (participant-level)

lr_fold_results = []
lr_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
lr_oof_labels = participant_labels.astype(np.int64).copy()

lr_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLD_SPLITS):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} (Logistic Regression) ---')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    # Recording-level features
    X_train_rec = features[train_mask]
    y_train_rec = y_raw[train_mask]
    parts_train_rec = participants_raw[train_mask]

    X_val_rec = features[val_mask]
    y_val_rec = y_raw[val_mask]
    parts_val_rec = participants_raw[val_mask]

    # Aggregate to participant level by averaging features across recordings
    def aggregate_by_participant(X, y, parts):
        unique_p = np.unique(parts)
        X_agg, y_agg = [], []
        for p in unique_p:
            mask = parts == p
            X_agg.append(X[mask].mean(axis=0))
            y_agg.append(y[mask][0])
        return np.array(X_agg), np.array(y_agg), unique_p

    X_train_part, y_train_part, train_pids = aggregate_by_participant(
        X_train_rec, y_train_rec, parts_train_rec)
    X_val_part, y_val_part, val_pids = aggregate_by_participant(
        X_val_rec, y_val_rec, parts_val_rec)

    print(f'  Train: {len(X_train_part)} participants, Val: {len(X_val_part)} participants')

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_part)
    X_val_scaled = scaler.transform(X_val_part)

    # Fit logistic regression
    clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    clf.fit(X_train_scaled, y_train_part)

    # Predict
    val_probs = clf.predict_proba(X_val_scaled)[:, 1]

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx = np.where(unique_participants == pid)[0][0]
        lr_oof_probs[idx] = val_probs[i]

    # Fold metrics
    fold_metrics = compute_fold_metrics(val_probs, y_val_part)
    fold_metrics['fold'] = fold + 1
    lr_fold_results.append(fold_metrics)

    print(f'  AUC={fold_metrics["auc"]:.4f}, F1={fold_metrics["f1"]:.4f}, '
          f'Rec={fold_metrics["recall"]:.4f}, Prec={fold_metrics["precision"]:.4f}')

lr_total_time = time.time() - lr_start
print(f'\nTotal LR time: {lr_total_time:.1f}s')

# Summary
print_cv_summary('LOGISTIC REGRESSION (SPECTROGRAM STATS)', lr_fold_results,
                 lr_oof_labels, lr_oof_probs)

# Save results
np.savez(str(RESULTS_DIR / 'lr_baseline_v3_cv_results.npz'),
    oof_probs=lr_oof_probs,
    oof_labels=lr_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in lr_fold_results]),
    fold_f1s=np.array([r['f1'] for r in lr_fold_results]),
    fold_recalls=np.array([r['recall'] for r in lr_fold_results]),
    fold_precisions=np.array([r['precision'] for r in lr_fold_results]),
    fold_thresholds=np.array([r['threshold'] for r in lr_fold_results]),
)
print(f'Saved: {RESULTS_DIR}/lr_baseline_v3_cv_results.npz')

---
## Baseline 3: ResNet18 (ImageNet Pretrained, Single-Channel)

Standard `torchvision.models.resnet18` modified for single-channel spectrogram input.
Conv1 weights initialized by averaging the 3-channel ImageNet weights.
Single learning rate 1e-4 (no differential LR). Input: (1, 128, 1024).

In [ ]:
#12 - ResNet18 model definition
from torchvision import models

class ResNet18Classifier(nn.Module):
    """ResNet18 adapted for single-channel spectrogram classification."""
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.resnet = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        )
        # Modify first conv for 1-channel input
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Average the 3-channel pretrained weights into 1 channel
        if pretrained:
            pretrained_conv1 = models.resnet18(
                weights=models.ResNet18_Weights.IMAGENET1K_V1
            ).conv1.weight.data
            self.resnet.conv1.weight.data = pretrained_conv1.mean(dim=1, keepdim=True)
        # Replace FC head
        in_features = self.resnet.fc.in_features  # 512
        self.resnet.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: (B, H, W) -> (B, 1, H, W)
        if x.dim() == 3:
            x = x.unsqueeze(1)
        return self.resnet(x)

# Print param count
_tmp = ResNet18Classifier(pretrained=True)
total_p = sum(p.numel() for p in _tmp.parameters())
trainable_p = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'ResNet18 params: {trainable_p:,} trainable / {total_p:,} total')
del _tmp

In [ ]:
#13 - ResNet18: 5-fold CV

resnet_fold_results = []
resnet_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
resnet_oof_labels = participant_labels.astype(np.int64).copy()

resnet_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLD_SPLITS):
    print(f'\n{"="*60}')
    print(f'Fold {fold+1}/{N_FOLDS} (ResNet18)')
    print(f'{"="*60}')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    print(f'Train: {train_mask.sum()} recordings from {len(train_parts_fold)} participants')
    print(f'Val:   {val_mask.sum()} recordings from {len(val_parts_fold)} participants')

    # Resize spectrograms to 128x1024
    X_train_res = np.stack([resize_spectrogram(x) for x in tqdm(X_raw[train_mask], desc='resize train', leave=False)])
    X_val_res = np.stack([resize_spectrogram(x) for x in tqdm(X_raw[val_mask], desc='resize val', leave=False)])

    # Z-score normalization (fold-specific)
    fold_mean = X_train_res.mean()
    fold_std = X_train_res.std()
    X_train_res = (X_train_res - fold_mean) / (fold_std + 1e-8)
    X_val_res = (X_val_res - fold_mean) / (fold_std + 1e-8)

    # Datasets with augmentation
    train_ds = SpectrogramDataset(X_train_res, y_raw[train_mask], participants_raw[train_mask], augment=True)
    val_ds = SpectrogramDataset(X_val_res, y_raw[val_mask], participants_raw[val_mask], augment=False)

    # Balanced sampler
    cc = np.bincount(y_raw[train_mask])
    sample_weights = (1.0 / cc)[y_raw[train_mask]]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler,
                                               num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False,
                                             num_workers=4, pin_memory=True)

    # Fresh model
    model = ResNet18Classifier(num_classes=2, pretrained=True).to(device)

    # Single LR 1e-4, no differential LR
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    criterion = FocalLoss(alpha=torch.tensor(cw, dtype=torch.float32).to(device), gamma=2.0)
    print(f'Class weights: {cw}')

    best_score = 0
    best_state = None
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(30):
        model.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels_batch = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        metrics = compute_fold_metrics(part_probs, part_labels_fold)
        score = 0.4 * metrics['auc'] + 0.6 * metrics['f1']

        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '<-- best'
        else:
            patience_counter += 1
            marker = ''

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {metrics["auc"]:.3f} | '
              f'F1 {metrics["f1"]:.3f} {marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and get final predictions
    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        resnet_oof_probs[idx_oof] = part_probs[i]

    # Fold metrics
    fold_metrics = compute_fold_metrics(part_probs, part_labels_fold)
    fold_metrics['fold'] = fold + 1
    resnet_fold_results.append(fold_metrics)

    print(f'Fold {fold+1}: AUC={fold_metrics["auc"]:.4f}, F1={fold_metrics["f1"]:.4f}')

    del model, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

resnet_total_time = time.time() - resnet_start
print(f'\nTotal ResNet18 CV time: {resnet_total_time/60:.1f} minutes')

# Summary
print_cv_summary('RESNET18 (ImageNet pretrained, 1-channel)', resnet_fold_results,
                 resnet_oof_labels, resnet_oof_probs)

# Save results
np.savez(str(RESULTS_DIR / 'resnet18_v3_cv_results.npz'),
    oof_probs=resnet_oof_probs,
    oof_labels=resnet_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in resnet_fold_results]),
    fold_f1s=np.array([r['f1'] for r in resnet_fold_results]),
    fold_recalls=np.array([r['recall'] for r in resnet_fold_results]),
    fold_precisions=np.array([r['precision'] for r in resnet_fold_results]),
    fold_thresholds=np.array([r['threshold'] for r in resnet_fold_results]),
)
print(f'Saved: {RESULTS_DIR}/resnet18_v3_cv_results.npz')

---
## Baseline 4: SpecAugment Ablation

Same fine-tuned AST pipeline as the primary experiment but with `augment=False`
(no time/frequency masking during training). This isolates the contribution of
SpecAugment to the primary result.

In [ ]:
#14 - SpecAugment ablation: 5-fold CV (NO augmentation)

ablation_fold_results = []
ablation_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
ablation_oof_labels = participant_labels.astype(np.int64).copy()

ablation_start = time.time()

for fold, (train_idx, val_idx) in enumerate(FOLD_SPLITS):
    print(f'\n{"="*60}')
    print(f'Fold {fold+1}/{N_FOLDS} (Fine-tuned AST — NO SpecAugment)')
    print(f'{"="*60}')

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]
    y_train_fold = y_raw[train_mask]
    parts_train_fold = participants_raw[train_mask]

    X_val_fold = X_raw[val_mask]
    y_val_fold = y_raw[val_mask]
    parts_val_fold = participants_raw[val_mask]

    print(f'Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants')
    print(f'Val:   {len(X_val_fold)} recordings from {len(val_parts_fold)} participants')

    # Resize 60 -> 128 mel bins
    X_train_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_train_fold, desc='resize train', leave=False)])
    X_val_ast = np.stack([resize_spectrogram(x) for x in tqdm(X_val_fold, desc='resize val', leave=False)])

    # Fold-specific z-score normalization
    fold_mean = X_train_ast.mean()
    fold_std = X_train_ast.std()
    X_train_ast = (X_train_ast - fold_mean) / (fold_std + 1e-8)
    X_val_ast = (X_val_ast - fold_mean) / (fold_std + 1e-8)

    # KEY DIFFERENCE: augment=False for BOTH train and val
    train_ds = SpectrogramDataset(X_train_ast, y_train_fold, parts_train_fold, augment=False)
    val_ds = SpectrogramDataset(X_val_ast, y_val_fold, parts_val_fold, augment=False)

    # Balanced sampler
    class_counts = np.bincount(y_train_fold)
    weights = 1.0 / class_counts
    sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler,
                                               num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False,
                                             num_workers=4, pin_memory=True)

    # Fresh model (FULL fine-tuning, same architecture as primary)
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)')

    # Differential LR (same as primary experiment)
    backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 5e-4, 'weight_decay': 0.01}
    ], betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights
    cc = np.bincount(y_train_fold)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = FocalLoss(alpha=class_weights_fold, gamma=2.0)
    print(f'Class weights: {cw}')

    best_score = 0
    best_state = None
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(30):
        model.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels_batch = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model, val_loader, device)
        metrics = compute_fold_metrics(part_probs, part_labels_fold)
        score = 0.4 * metrics['auc'] + 0.6 * metrics['f1']

        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            marker = '<-- best'
        else:
            patience_counter += 1
            marker = ''

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch+1:02d} | loss {avg_loss:.4f} | AUC {metrics["auc"]:.3f} | '
              f'F1 {metrics["f1"]:.3f} {marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best and get final predictions
    model.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model, val_loader, device)

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx_oof = np.where(unique_participants == pid)[0][0]
        ablation_oof_probs[idx_oof] = part_probs[i]

    # Fold metrics
    fold_metrics = compute_fold_metrics(part_probs, part_labels_fold)
    fold_metrics['fold'] = fold + 1
    ablation_fold_results.append(fold_metrics)

    print(f'Fold {fold+1}: AUC={fold_metrics["auc"]:.4f}, F1={fold_metrics["f1"]:.4f}')

    del model, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

ablation_total_time = time.time() - ablation_start
print(f'\nTotal SpecAugment ablation CV time: {ablation_total_time/60:.1f} minutes')

# Summary
print_cv_summary('SPECAUGMENT ABLATION (Fine-tuned AST, no augment)', ablation_fold_results,
                 ablation_oof_labels, ablation_oof_probs)

# Save results
np.savez(str(RESULTS_DIR / 'specaugment_ablation_v3_cv_results.npz'),
    oof_probs=ablation_oof_probs,
    oof_labels=ablation_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in ablation_fold_results]),
    fold_f1s=np.array([r['f1'] for r in ablation_fold_results]),
    fold_recalls=np.array([r['recall'] for r in ablation_fold_results]),
    fold_precisions=np.array([r['precision'] for r in ablation_fold_results]),
    fold_thresholds=np.array([r['threshold'] for r in ablation_fold_results]),
)
print(f'Saved: {RESULTS_DIR}/specaugment_ablation_v3_cv_results.npz')

---
## Summary: Baseline Comparison Table

Compares all four baselines plus the fine-tuned AST primary experiment (loaded from saved results if available).

In [ ]:
#15 - Final comparison table
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

# Collect all baseline results
all_results = {}

# Try to load fine-tuned AST primary results
ft_path = RESULTS_DIR / 'ast_pd_v3_cv_results.npz'
if ft_path.exists():
    ft = np.load(str(ft_path), allow_pickle=True)
    ft_probs = ft['oof_probs']
    ft_labels = ft['oof_labels']
    ft_auc = roc_auc_score(ft_labels, ft_probs)
    fpr_ft, tpr_ft, thr_ft = roc_curve(ft_labels, ft_probs)
    opt_ft = np.argmax(tpr_ft - fpr_ft)
    ft_preds = (ft_probs >= thr_ft[opt_ft]).astype(int)
    all_results['Fine-tuned AST'] = {
        'oof_auc': ft_auc,
        'oof_f1': f1_score(ft_labels, ft_preds, zero_division=0),
        'oof_recall': recall_score(ft_labels, ft_preds, zero_division=0),
        'oof_precision': precision_score(ft_labels, ft_preds, zero_division=0),
        'fold_aucs': ft['fold_aucs'],
        'fold_f1s': ft['fold_f1s'],
        'probs': ft_probs,
        'labels': ft_labels,
    }
    print(f'Loaded fine-tuned AST results from {ft_path}')
else:
    print(f'Fine-tuned AST results not found at {ft_path} — will be omitted from comparison.')

# Add baselines computed in this notebook
for name, fold_res, oof_probs, oof_labels in [
    ('Frozen AST (linear probe)', frozen_fold_results, frozen_oof_probs, frozen_oof_labels),
    ('Logistic Regression', lr_fold_results, lr_oof_probs, lr_oof_labels),
    ('ResNet18 (ImageNet)', resnet_fold_results, resnet_oof_probs, resnet_oof_labels),
    ('AST w/o SpecAugment', ablation_fold_results, ablation_oof_probs, ablation_oof_labels),
]:
    oof_auc = roc_auc_score(oof_labels, oof_probs)
    fpr_t, tpr_t, thr_t = roc_curve(oof_labels, oof_probs)
    opt_t = np.argmax(tpr_t - fpr_t)
    preds_t = (oof_probs >= thr_t[opt_t]).astype(int)
    all_results[name] = {
        'oof_auc': oof_auc,
        'oof_f1': f1_score(oof_labels, preds_t, zero_division=0),
        'oof_recall': recall_score(oof_labels, preds_t, zero_division=0),
        'oof_precision': precision_score(oof_labels, preds_t, zero_division=0),
        'fold_aucs': np.array([r['auc'] for r in fold_res]),
        'fold_f1s': np.array([r['f1'] for r in fold_res]),
        'probs': oof_probs,
        'labels': oof_labels,
    }

# Print comparison table
print(f'\n{"="*90}')
print(f'BASELINE COMPARISON TABLE — v3.0.0 PD Selected Tasks (Participant-Level OOF Metrics)')
print(f'{"="*90}')
print(f'{"Model":<30} {"OOF AUC":>8} {"Mean AUC (SD)":>16} {"OOF F1":>8} '
      f'{"OOF Recall":>11} {"OOF Prec":>9}')
print(f'{"-"*90}')

for name, res in all_results.items():
    mean_auc = res['fold_aucs'].mean()
    std_auc = res['fold_aucs'].std(ddof=1)
    print(f'{name:<30} {res["oof_auc"]:>8.3f} {mean_auc:>8.3f} ({std_auc:.3f}) '
          f'{res["oof_f1"]:>8.3f} {res["oof_recall"]:>11.3f} {res["oof_precision"]:>9.3f}')

print(f'{"-"*90}')

# Interpretation
if 'Fine-tuned AST' in all_results:
    ft_auc = all_results['Fine-tuned AST']['oof_auc']
    print(f'\nDelta AUC vs Fine-tuned AST:')
    for name, res in all_results.items():
        if name != 'Fine-tuned AST':
            delta = res['oof_auc'] - ft_auc
            print(f'  {name}: {delta:+.3f} AUC')

In [ ]:
#16 - ROC comparison plot
fig, ax = plt.subplots(1, 1, figsize=(8, 7))

styles = {
    'Fine-tuned AST': ('b-', 2.5),
    'Frozen AST (linear probe)': ('r--', 2.0),
    'Logistic Regression': ('g-.', 2.0),
    'ResNet18 (ImageNet)': ('m:', 2.0),
    'AST w/o SpecAugment': ('c--', 1.5),
}

for name, res in all_results.items():
    style, lw = styles.get(name, ('k-', 1.0))
    fpr_p, tpr_p, _ = roc_curve(res['labels'], res['probs'])
    ax.plot(fpr_p, tpr_p, style, linewidth=lw,
            label=f'{name} (AUC={res["oof_auc"]:.3f})')

ax.plot([0, 1], [0, 1], 'k:', linewidth=1, alpha=0.4)
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('v3.0.0 PD Selected Tasks: Baseline Comparison (OOF)', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = RESULTS_DIR / 'v3_baseline_roc_comparison.png'
plt.savefig(str(fig_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
#17 - Per-fold AUC comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))

model_names = list(all_results.keys())
n_models = len(model_names)
x = np.arange(N_FOLDS)
width = 0.8 / n_models

colors = ['#2196F3', '#F44336', '#4CAF50', '#9C27B0', '#00BCD4']

for i, (name, res) in enumerate(all_results.items()):
    offset = (i - n_models / 2 + 0.5) * width
    bars = ax.bar(x + offset, res['fold_aucs'], width * 0.9,
                  label=name, color=colors[i % len(colors)], alpha=0.8)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Per-Fold AUC Comparison Across Baselines', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(N_FOLDS)])
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.05])

plt.tight_layout()
fig_path2 = RESULTS_DIR / 'v3_baseline_fold_comparison.png'
plt.savefig(str(fig_path2), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path2}')

In [ ]:
#18 - Final summary with timing
print(f'\n{"="*70}')
print('EXPERIMENT COMPLETE — v3.0.0 Baseline Comparisons')
print(f'{"="*70}')
print(f'\nResults saved to: {RESULTS_DIR}')
print(f'  - frozen_ast_v3_cv_results.npz')
print(f'  - lr_baseline_v3_cv_results.npz')
print(f'  - resnet18_v3_cv_results.npz')
print(f'  - specaugment_ablation_v3_cv_results.npz')
print(f'  - v3_baseline_roc_comparison.png')
print(f'  - v3_baseline_fold_comparison.png')
print(f'\nTiming:')
print(f'  Frozen AST:           {frozen_total_time/60:.1f} min')
print(f'  Logistic Regression:  {lr_total_time:.1f} sec')
print(f'  ResNet18:             {resnet_total_time/60:.1f} min')
print(f'  SpecAugment ablation: {ablation_total_time/60:.1f} min')
total_all = frozen_total_time + lr_total_time + resnet_total_time + ablation_total_time
print(f'  Total:                {total_all/60:.1f} min')